# Module 3: Predicting Prospect Translation

**Question:** Can we predict which top prospects will convert tools into production using pitch-level Statcast data from their first ~200 PA?

We build two models:
1. **XGBoost** — for feature importance (which metrics matter most?)
2. **Bayesian logistic regression** (Bambi/PyMC) — for interpretable effect sizes with uncertainty

Then apply both to Volpe and Dominguez's current data.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from fire_fishman.data.statcast import get_statcast_pitches, get_batting_stats
from fire_fishman.data.prospects import get_prospect_df
from fire_fishman.features.tools_score import compute_tools_for_cohort
from fire_fishman.features.pitch_level import compute_pitch_features_for_cohort

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 7)

In [ ]:
# Load cached data
pitches_2023 = get_statcast_pitches(2023)
pitches_2024 = get_statcast_pitches(2024)
pitches = pd.concat([pitches_2023, pitches_2024], ignore_index=True)

prospects = get_prospect_df()
batter_ids = prospects["mlbam_id"].tolist()

## 1. Build Feature Matrix

Combine tools scores + pitch-level features into a single feature matrix. Target: binary "success" (star or solid) vs. "failure" (disappointing or bust).

In [ ]:
# Compute features
tools_df = compute_tools_for_cohort(pitches, batter_ids)
pitch_df = compute_pitch_features_for_cohort(pitches, batter_ids)

# Merge everything
feature_df = tools_df.join(pitch_df, how="inner")
feature_df = feature_df.join(prospects.set_index("mlbam_id")[["name", "outcome", "pre_debut_rank"]])

# Binary target
feature_df["success"] = feature_df["outcome"].isin(["star", "solid"]).astype(int)

print(f"Feature matrix: {feature_df.shape}")
print(f"\nSuccess rate: {feature_df['success'].mean():.1%}")
print(f"Stars/Solid: {feature_df['success'].sum()} | Disappointing/Bust: {(~feature_df['success'].astype(bool)).sum()}")

In [ ]:
# Define feature columns (drop metadata and z-scores to avoid leakage)
exclude_cols = ["name", "outcome", "success", "total_pitches_seen"]
exclude_cols += [c for c in feature_df.columns if c.endswith("_z")]

feature_cols = [c for c in feature_df.columns
                if c not in exclude_cols and feature_df[c].dtype in ["float64", "int64"]]

print(f"Using {len(feature_cols)} features:")
for c in feature_cols:
    print(f"  - {c}")

## 2. XGBoost — Feature Importance

With a small sample (~20 prospects), we use leave-one-out cross-validation and focus on feature importance rather than predictive accuracy.

In [ ]:
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

X = feature_df[feature_cols].copy()
y = feature_df["success"].values

# Impute missing values
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

# LOO-CV
loo = LeaveOneOut()
predictions = []
probabilities = []
importances_list = []

for train_idx, test_idx in loo.split(X_imputed):
    X_train, X_test = X_imputed.iloc[train_idx], X_imputed.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Conservative hyperparams for small sample
    model = XGBClassifier(
        n_estimators=50, max_depth=2, learning_rate=0.1,
        min_child_weight=3, subsample=0.8,
        use_label_encoder=False, eval_metric="logloss", verbosity=0,
    )
    model.fit(X_train, y_train)

    predictions.append(model.predict(X_test)[0])
    probabilities.append(model.predict_proba(X_test)[0, 1])
    importances_list.append(model.feature_importances_)

# Results
acc = accuracy_score(y, predictions)
try:
    auc = roc_auc_score(y, probabilities)
except ValueError:
    auc = float("nan")

print(f"LOO-CV Accuracy: {acc:.1%}")
print(f"LOO-CV AUC: {auc:.3f}")

In [ ]:
# Average feature importance across LOO folds
avg_importance = np.mean(importances_list, axis=0)
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": avg_importance,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(importance_df["feature"], importance_df["importance"], color="#3498db", alpha=0.8)
ax.set_xlabel("Average Feature Importance (XGBoost, LOO-CV)")
ax.set_title("What Predicts Prospect Success?\nFeature importance from XGBoost", fontsize=14)

plt.tight_layout()
plt.savefig("../outputs/figures/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTop 5 features:")
importance_df.tail(5)[["feature", "importance"]]

## 3. Bayesian Logistic Regression (Bambi)

Now the interpretable model. We use the top features from XGBoost to build a Bayesian logistic regression that gives us:
- Effect sizes with credible intervals
- Posterior predictive distributions for Volpe/Dominguez
- Honest uncertainty about predictions

In [ ]:
import bambi as bmb
import arviz as az

# Select top features from XGBoost (avoid overfitting with too many predictors)
top_features = importance_df.tail(5)["feature"].tolist()
print(f"Using top features for Bayesian model: {top_features}")

# Prepare model dataframe
model_df = feature_df[top_features + ["success", "name"]].dropna().copy()

# Standardize features for interpretable priors
for col in top_features:
    model_df[col] = (model_df[col] - model_df[col].mean()) / model_df[col].std()

print(f"Model sample size: {len(model_df)}")

In [ ]:
# Build formula
formula = "success ~ " + " + ".join(top_features)
print(f"Formula: {formula}")

# Fit Bayesian model with weakly informative priors
model = bmb.Model(
    formula,
    data=model_df,
    family="bernoulli",
    priors={
        "Intercept": bmb.Prior("Normal", mu=0, sigma=2),
        "common": bmb.Prior("Normal", mu=0, sigma=1),
    },
)

print(model)

In [ ]:
# Sample
idata = model.fit(
    draws=2000,
    tune=1000,
    chains=4,
    target_accept=0.9,
    random_seed=42,
)

# Diagnostics
print(az.summary(idata, var_names=["Intercept"] + top_features))

In [ ]:
# Forest plot of coefficients
fig, ax = plt.subplots(figsize=(10, 6))
az.plot_forest(idata, var_names=top_features, combined=True, ax=ax)
ax.axvline(0, color="red", linestyle="--", alpha=0.5)
ax.set_title("Bayesian Model: Effect Sizes on Prospect Success\n(log-odds scale, 94% HDI)", fontsize=14)

plt.tight_layout()
plt.savefig("../outputs/figures/bayesian_forest.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Predict Volpe & Dominguez

What does the model say about their probability of "figuring it out"?

In [ ]:
# Posterior predictive for Volpe and Dominguez
yankees = model_df[model_df["name"].isin(["Anthony Volpe", "Jasson Dominguez"])]

if len(yankees) > 0:
    ppc = model.predict(idata, data=yankees, kind="pps")

    fig, axes = plt.subplots(1, len(yankees), figsize=(12, 5))
    if len(yankees) == 1:
        axes = [axes]

    for ax, (idx, row) in zip(axes, yankees.iterrows()):
        # Extract posterior predictive probabilities
        # pps returns binary outcomes; we want the mean probability
        post_pred = idata.posterior_predictive["success"].sel({"success_obs": list(yankees.index).index(idx)})
        prob_success = post_pred.values.flatten().mean()

        ax.hist(post_pred.values.flatten(), bins=2, color="#3498db", alpha=0.7, edgecolor="white")
        ax.set_title(f"{row['name']}\nP(success) = {prob_success:.1%}", fontsize=13)
        ax.set_xlabel("Predicted Outcome")
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Bust/Disappointing", "Star/Solid"])

    plt.suptitle("Model Predictions for Yankees Prospects", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("../outputs/figures/yankees_predictions.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Yankees prospects not found in model dataframe — check for NaN features.")

## 5. Model Diagnostics

In [ ]:
# Trace plots
az.plot_trace(idata, var_names=top_features)
plt.tight_layout()
plt.show()

# R-hat check
summary = az.summary(idata, var_names=top_features)
max_rhat = summary["r_hat"].max()
print(f"\nMax R-hat: {max_rhat:.4f} {'✓ Good' if max_rhat < 1.01 else '✗ Convergence issue'}")

# LOO-CV for model comparison
loo = az.loo(idata)
print(f"\nLOO-CV ELPD: {loo.elpd_loo:.2f} (SE: {loo.se:.2f})")
print(f"Pareto k diagnostic: {(loo.pareto_k > 0.7).sum()} problematic observations")

## Key Findings

Summarize:
1. Which features most predict prospect success/failure?
2. What does the model say about Volpe and Dominguez specifically?
3. How confident is the model (uncertainty)?
4. Caveats: small sample, potential confounders

→ Continue to **Notebook 04** for actionable prescriptions.